# 1. import + 파일 불러오기

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 현재 노트북이 works/Hyeong_Uk/shorts_video_analysis 안에 있다고 가정
RESULT_PATH = Path("results/result_sample_shorts_all_for_video_agent.csv")
ORIGIN_PATH = Path("data/fnb_shorts_success_data.csv")

df = pd.read_csv(RESULT_PATH, encoding="utf-8-sig")
origin = pd.read_csv(ORIGIN_PATH, encoding="utf-8-sig")

print("분석 결과 df:", df.shape)
print("원본 origin:", origin.shape)

display(df.head())

분석 결과 df: (200, 75)
원본 origin: (384, 56)


,video_id,production_quality,lighting_style,color_mood,editing_pace,motion_graphic,video_format,first_3sec,background_style,top_colors,...,좋아요성과,댓글성과,조회수성과_상위1%,조회수성과_상위5%,좋아요성과_상위1%,좋아요성과_상위5%,댓글성과_상위1%,댓글성과_상위5%,score2,grade
0,VbWATcKlZ2Y,프로페셔널,인공조명,비비드,매우 빠름,핵심요소,애니메이션,인물,스튜디오,"['노란색', '흰색', '빨간색']",...,0.943130,0.409459,0.0,0.0,0.0,0.0,0.0,0.0,0.723787,실패
1,VxMVRz48OxU,프로페셔널,인공조명,중립,빠름,보조적,광고/CF,기업 로고,실내,"['회색', '녹색', '검은색']",...,0.828733,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.439959,실패
2,XXBuAxdGCf4,프로페셔널,인공조명,비비드,매우 빠름,보조적,광고/CF,인물,실내,"['빨간색', '흰색', '파란색']",...,0.618543,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.578244,실패
3,ddRC6B2nPpo,프로페셔널,인공조명,따뜻함,보통,핵심요소,광고/CF,텍스트,실내,"['빨간색', '검은색', '흰색']",...,0.700676,0.471514,0.0,0.0,0.0,0.0,0.0,0.0,0.525058,실패
4,VL2FgtMmI3g,프로페셔널,인공조명,따뜻함,보통,보조적,광고/CF,텍스트,실내,"['갈색', '금색', '흰색']",...,0.423689,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.455503,실패


# 2. 잘못된 video_id 수정
- Gemini가 video_id를 한 글자 잘못 복사해서 반환한 문제.
- 실제로 없는 영상이 생긴 게 아니라, 원래 pnz7WKfQw58 영상의 분석 결과가 pnz7WKQw58라는 잘못된 video_id로 저장된 것으로 보면됨.

기존 agent 코드에 combined["video_id"] = video_id를 추가해줘야 함

In [2]:
wrong_id = "pnz7WKQw58"
correct_id = "pnz7WKfQw58"

print("수정 전 wrong_id 개수:", (df["video_id"] == wrong_id).sum())
print("수정 전 correct_id 개수:", (df["video_id"] == correct_id).sum())

df.loc[df["video_id"] == wrong_id, "video_id"] = correct_id

print("수정 후 wrong_id 개수:", (df["video_id"] == wrong_id).sum())
print("수정 후 correct_id 개수:", (df["video_id"] == correct_id).sum())

수정 전 wrong_id 개수: 1
수정 전 correct_id 개수: 0
수정 후 wrong_id 개수: 0
수정 후 correct_id 개수: 1


# 3. 원본에서 해당 video_id 정보 확인

In [3]:
origin["video_id"] = origin["video_id"].astype(str).str.strip()
df["video_id"] = df["video_id"].astype(str).str.strip()

origin_row = origin[origin["video_id"] == correct_id]

print("원본에서 찾은 행 수:", len(origin_row))
display(origin_row)

원본에서 찾은 행 수: 1


,video_id,title,channel_id,채널명,description,업로드일시,tags,조회수,좋아요수,댓글수,...,좋아요성과,댓글성과,조회수성과_상위1%,조회수성과_상위5%,좋아요성과_상위1%,좋아요성과_상위5%,댓글성과_상위1%,댓글성과_상위5%,score2,grade
3,pnz7WKfQw58,CU 간편식 피빅더키친 시리즈 ASMR | [톡! 까놓고 신상리뷰],UCsM07dUwo0WOWhFbVsG8K6A,CU [씨유튜브],귀로 먼저 먹는 CU 간편식 ASMR 🎧\n배우 이송경 X 유튜버 맛상무가\n톡! ...,2026-03-19 08:42:10,"CU편의점, CU, 씨유, 헤이루, Heyroo",242162,9732.0,15.0,...,5.056364,4.123604,0,0,1,1,0,0,3.470222,성공


# 4. 결측 난 메타데이터 컬럼만 보완

In [4]:
# df에 있고 origin에도 있는 컬럼 중, video_id는 제외
common_cols = [col for col in df.columns if col in origin.columns and col != "video_id"]

print("보완 가능한 공통 컬럼 수:", len(common_cols))
print(common_cols)

보완 가능한 공통 컬럼 수: 55
['채널명', 'domain', 'title', 'channel_id', 'description', '업로드일시', 'tags', '조회수', '좋아요수', '댓글수', '영상길이(초)', 'definition', 'license', 'embeddable', 'has_paid_product_placement', 'thumbnail', 'caption', 'final_url', 'instream_type', 'channel_handle', 'channel_tier', '구독자수', 'description_missing_flag', 'tags_missing_flag', '참여율(ER)', '조회수 대비 좋아요율', '조회수 대비 댓글률', 'wei', 'description_length', 'category_name', 'upload_year', 'upload_month', 'upload_dayofweek', 'upload_hour', 'tags_count', 'upload_quarter', 'upload_ym_quarter', 'upload_ymd', '경과일수', '도달률(RR)', '일평균 조회수', 'RR_백분위', 'ER_백분위', 'score1', '조회수성과', '좋아요성과', '댓글성과', '조회수성과_상위1%', '조회수성과_상위5%', '좋아요성과_상위1%', '좋아요성과_상위5%', '댓글성과_상위1%', '댓글성과_상위5%', 'score2', 'grade']


In [5]:
if len(origin_row) == 1:
    origin_one = origin_row.iloc[0]

    target_idx = df[df["video_id"] == correct_id].index

    print("수정 대상 행 수:", len(target_idx))

    for col in common_cols:
        # df의 값이 비어 있는 경우에만 origin 값으로 채움
        df.loc[target_idx, col] = df.loc[target_idx, col].fillna(origin_one[col])

    display(df.loc[target_idx])
else:
    print("원본에서 해당 video_id를 1개로 찾지 못했습니다.")

수정 대상 행 수: 1


,video_id,production_quality,lighting_style,color_mood,editing_pace,motion_graphic,video_format,first_3sec,background_style,top_colors,...,좋아요성과,댓글성과,조회수성과_상위1%,조회수성과_상위5%,좋아요성과_상위1%,좋아요성과_상위5%,댓글성과_상위1%,댓글성과_상위5%,score2,grade
92,pnz7WKfQw58,고퀄리티,인공조명,따뜻함,빠름,보조적,제품리뷰,인물,스튜디오,"['베이지색', '흰색', '파란색']",...,5.056364,4.123604,0.0,0.0,1.0,1.0,0.0,0.0,3.470222,성공


# 5. 다시 결측 확인

In [6]:
missing_summary = (
    df.isna()
      .sum()
      .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_ratio"] = (missing_summary["missing_count"] / len(df)).round(3)

missing_summary = missing_summary.sort_values("missing_count", ascending=False)

display(missing_summary[missing_summary["missing_count"] > 0])

,column,missing_count,missing_ratio
66,댓글성과,14,0.07


# 6. 수정본 저장

In [7]:
FIXED_PATH = Path("results/result_sample_shorts_all_for_video_agent_fixed.csv")

df.to_csv(FIXED_PATH, index=False, encoding="utf-8-sig")

print("저장 완료:", FIXED_PATH)
print("최종 데이터 크기:", df.shape)

저장 완료: results\result_sample_shorts_all_for_video_agent_fixed.csv
최종 데이터 크기: (200, 75)
